# Project 1 V1


In [ ]:
# Install dependencies (Google Colab only; local/NERSC runs use the project environment).
import sys

if "google.colab" in sys.modules:
    %pip install -q "numpy>=2,<3" tensorflow


## Load the Data

The extended files contain the provided rows first, followed by the generated rows. `N_PROVIDED`, the number of provided rows, marks the boundary. The provided rows are the only ones used for fine-tuning and for model selection.


In [ ]:
import os
import pickle
import numpy as np

NUMPY_MAJOR = int(np.__version__.split(".")[0])


class NumpyCompatUnpickler(pickle.Unpickler):
    # NumPy 2 pickles reference numpy._core; NumPy 1.x only provides numpy.core.
    def find_class(self, module, name):
        if NUMPY_MAJOR < 2 and module.startswith("numpy._core"):
            module = "numpy.core" + module[len("numpy._core"):]
        return super().find_class(module, name)


def load_pickle(path):
    with open(path, "rb") as file:
        return NumpyCompatUnpickler(file).load()


# Pre-training uses the extended data; its first N_PROVIDED rows are the provided data.
DATASET = "extended"
features = load_pickle("data/CSCE-636-Project-1-Train-Extended-n_k_m_P")
labels = load_pickle("data/CSCE-636-Project-1-Train-Extended-mHeights")
print(f"Dataset: {DATASET}")

N_PROVIDED = len(load_pickle("data/CSCE-636-Project-1-Train-mHeights"))

if len(features) != len(labels):
    raise ValueError(
        f"Sample/label count mismatch: {len(features)} samples, {len(labels)} labels."
    )

first_five_features = features[:5]
first_five_labels = labels[:5]
print(f"Loaded {len(features):,} samples and {len(labels):,} corresponding labels.")

# Print the first five samples and their corresponding labels.
for index, ((n, k, m, P), m_height) in enumerate(
    zip(first_five_features, first_five_labels)
):
    if n != 9 or k not in (4, 5, 6) or not (2 <= m <= n - k):
        raise ValueError(f"Sample at index {index} has invalid n, k, or m.")
    if not isinstance(P, np.ndarray) or P.shape != (k, n - k):
        raise ValueError(f"Sample at index {index}: expected a NumPy array of shape {(k, n - k)}.")

    print("=" * 64)
    print(f"Sample {index + 1} (list index {index})")
    print(f"n = {n}, k = {k}, m = {m}")
    print(f"P.shape = {P.shape}; expected (k, n-k) = {(k, n - k)}")
    print(f"P dtype: {P.dtype}")
    print("P contents:")
    with np.printoptions(threshold=np.inf, floatmode="unique", linewidth=120):
        print(P)
    print(f"True m-height = {m_height!r}")



## Configuration

The network predicts $z=\log_2(\hat y)$, bounded to `[0, 100]`, and $\hat y=2^z$.

| Setting | Default | Environment variable |
|---|---|---|
| Pre-training epochs | 50 | `MHEIGHT_PRETRAIN_EPOCHS` |
| Fine-tuning epochs | 50 | `MHEIGHT_FINETUNE_EPOCHS` |
| Fine-tuning learning rate | 3e-4 | `MHEIGHT_FINETUNE_LR` |
| Output directory | `artifacts/dnn_v1_finetune` | `MHEIGHT_RUN` |


In [ ]:
import hashlib
import os
import json
import pickle
import numpy as np
from pathlib import Path
import tensorflow as tf

SEED = 636
BATCH_SIZE = 4096
INFERENCE_BATCH_SIZE = 1024
PRETRAIN_EPOCHS = int(os.environ.get("MHEIGHT_PRETRAIN_EPOCHS", 50))
FINETUNE_EPOCHS = int(os.environ.get("MHEIGHT_FINETUNE_EPOCHS", 50))
FINETUNE_LEARNING_RATE = float(os.environ.get("MHEIGHT_FINETUNE_LR", 3e-4))
MAX_LOG2_HEIGHT = 100.0
ARTIFACT_DIR = Path("artifacts") / os.environ.get("MHEIGHT_RUN", "dnn_v1_finetune")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
PRETRAINED_PATH = ARTIFACT_DIR / "pretrained_model.keras"
# The fine-tuned model is the final model.
MODEL_PATH = ARTIFACT_DIR / "best_model.keras"
PARAMETER_SETTINGS = [(9, k, m) for k in (4, 5, 6) for m in range(2, 9 - k + 1)]

# Set TensorFlow to be deterministic and reproducible.
tf.keras.utils.set_random_seed(SEED)

# Enable deterministic operations in TensorFlow to ensure reproducibility.
tf.config.experimental.enable_op_determinism()

# Enable GPU memory growth to avoid allocating all GPU memory at once.
for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)

# Print TensorFlow version and available GPUs.
print("TensorFlow:", tf.__version__, "| Keras:", tf.keras.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))


## Input Formatting and Data Splitting

`P` varies in shape, so we place it in the upper-left corner of a zero-padded `6 x 5` array. The original entries and `[n, k, m]` are the only model inputs. Fixed scaling occurs inside standard Keras `Rescaling` layers.

The same matrix can appear under several `m` values. Hence, a row-wise shuffle could place that matrix in both training and testing. Therefore, a seeded hash of `(n, k, P)` assigns each distinct matrix to training, validation, or testing with probabilities **60%, 20%, 20%**. Identical matrix contents always get the same assignment, even if stored as different NumPy objects. Actual row percentages are approximate because whole matrix groups stay together. The hash is used only for partitioning and is never fed into the model.

Nonfinite or below-one training labels have no valid finite log-loss target. We report and exclude such rows from supervised training and local evaluation.

In [ ]:
def pack_features(samples):
    matrices = np.zeros((len(samples), 6, 5), dtype=np.float32)
    parameters = np.empty((len(samples), 3), dtype=np.float32)
    for index, (n, k, m, P) in enumerate(samples):
        if n != 9 or k not in (4, 5, 6) or not 2 <= m <= n - k:
            raise ValueError(f"Invalid parameters at row {index}: {(n, k, m)}")
        if not isinstance(P, np.ndarray) or P.shape != (k, n - k):
            raise ValueError(f"Invalid P shape at row {index}")
        matrices[index, :k, :n - k] = P
        parameters[index] = (n, k, m)
    if not np.isfinite(matrices).all():
        raise ValueError("Input matrices must remain finite after float32 conversion")
    return {"matrix": matrices, "parameters": parameters}


def matrix_partitions(samples, seed):
    partitions = np.empty(len(samples), dtype=np.uint8)
    object_cache = {}
    key = int(seed).to_bytes(8, "little")
    for index, (n, k, m, P) in enumerate(samples):
        cache_key = (int(n), int(k), id(P))
        if cache_key not in object_cache:
            payload = bytes((int(n), int(k))) + np.asarray(P, dtype="<f8").tobytes(order="C")
            digest = hashlib.blake2b(payload, digest_size=8, key=key).digest()
            bucket = int.from_bytes(digest, "little") % 10
            object_cache[cache_key] = 0 if bucket < 6 else (1 if bucket < 8 else 2)
        partitions[index] = object_cache[cache_key]
    return partitions


In [ ]:
if len(features) != len(labels):
    raise ValueError("Feature and label counts differ")
raw_targets = np.asarray(labels, dtype=np.float64)
if raw_targets.shape != (len(features),):
    raise ValueError("Expected one scalar label per sample")
valid_target = np.isfinite(raw_targets) & (raw_targets >= 1.0)
print(f"Excluded {np.count_nonzero(~valid_target):,} invalid training labels")
if not np.any(valid_target):
    raise ValueError("No valid training labels")

targets_log2 = np.zeros((len(features), 1), dtype=np.float32)
targets_log2[valid_target, 0] = np.log2(raw_targets[valid_target])
if targets_log2[valid_target].max() > MAX_LOG2_HEIGHT:
    raise ValueError("A training target exceeds the model output range")
inputs = pack_features(features)
partition = matrix_partitions(features, SEED)
train_indices = np.flatnonzero(valid_target & (partition == 0))
validation_indices = np.flatnonzero(valid_target & (partition == 1))
test_indices = np.flatnonzero(valid_target & (partition == 2))

setting_ids = np.empty(len(features), dtype=np.int8)
for group_id, (n, k, m) in enumerate(PARAMETER_SETTINGS):
    mask = (inputs["parameters"][:, 1] == k) & (inputs["parameters"][:, 2] == m)
    setting_ids[mask] = group_id

splits = {"train": train_indices, "validation": validation_indices, "test": test_indices}
for name, indices in splits.items():
    if len(indices) == 0:
        raise ValueError(f"The {name} split is empty")
    print(f"{name:10s}: {len(indices):,} rows ({100 * len(indices) / valid_target.sum():.2f}%)")
print("\n(n, k, m)       train       validation       test")
for group_id, setting in enumerate(PARAMETER_SETTINGS):
    counts = [int(np.count_nonzero(setting_ids[indices] == group_id)) for indices in splits.values()]
    if min(counts) == 0:
        raise ValueError(f"Setting {setting} is missing from a split")
    print(f"{str(setting):12s} {counts[0]:10,d} {counts[1]:14,d} {counts[2]:10,d}")
np.savez_compressed(ARTIFACT_DIR / "split_indices.npz", **splits)

# Provided-data rows of each split: used for fine-tuning, model selection and reporting.
provided_train_indices = train_indices[train_indices < N_PROVIDED]
provided_validation_indices = validation_indices[validation_indices < N_PROVIDED]
provided_test_indices = test_indices[test_indices < N_PROVIDED]
print(f"\nProvided-data rows: train {len(provided_train_indices):,}, "
      f"validation {len(provided_validation_indices):,}, test {len(provided_test_indices):,}")


## DNN

The Deep Nerual Network model has three hidden dense layers with 256, 128, and 64 ReLU units. The Softplus output is nonnegative, and the final bounded ReLU enforces the numerical output range. Fixed `P / 100` and parameter scaling are part of the saved model, so training and inference use the same transformation.

We weight training and validation rows inversely by their group's frequency so that equal weights are given to each parameter setting, even though the dataset has unequal group sizes.

In [ ]:
def build_model():
    matrix_input = tf.keras.Input(shape=(6, 5), name="matrix")
    parameter_input = tf.keras.Input(shape=(3,), name="parameters")
    matrix_values = tf.keras.layers.Rescaling(1.0 / 100.0)(matrix_input)
    matrix_values = tf.keras.layers.Flatten()(matrix_values)
    parameter_values = tf.keras.layers.Rescaling(1.0 / 10.0)(parameter_input)
    x = tf.keras.layers.Concatenate()([matrix_values, parameter_values])
    for units in (256, 128, 64):
        x = tf.keras.layers.Dense(units, activation="relu")(x)
    x = tf.keras.layers.Dense(1, activation="softplus")(x)
    output = tf.keras.layers.ReLU(max_value=MAX_LOG2_HEIGHT, name="log2_height")(x)
    network = tf.keras.Model(
        inputs={"matrix": matrix_input, "parameters": parameter_input},
        outputs=output, name="mheight_mlp",
    )
    network.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="mean_squared_error",
        # Sample weights already balance the loss; no additional metrics requested.
        weighted_metrics=[],
    )
    return network

model = build_model()
model.summary()


In [ ]:
def setting_weights(indices):
    counts = np.bincount(setting_ids[indices], minlength=len(PARAMETER_SETTINGS))
    if np.any(counts == 0):
        raise ValueError("Each parameter setting must be present in the weighted split")
    return (len(indices) / (len(PARAMETER_SETTINGS) * counts[setting_ids[indices]])).astype(np.float32)


def make_dataset(indices, training=False):
    # Keep the dataset on host memory; only batches move to the training device.
    with tf.device("/CPU:0"):
        dataset = tf.data.Dataset.from_tensor_slices((
            {name: values[indices] for name, values in inputs.items()},
            targets_log2[indices], setting_weights(indices),
        ))
    if training:
        dataset = dataset.shuffle(min(len(indices), 100_000), seed=SEED, reshuffle_each_iteration=True)
    options = tf.data.Options()
    options.experimental_deterministic = True
    options.threading.private_threadpool_size = 2
    return dataset.with_options(options).batch(BATCH_SIZE).prefetch(1)


## Phase 1: Pre-train on the Extended Data

Pre-training uses all extended training rows. Its validation set is all extended validation rows. The best weights are restored at the end and saved as `pretrained_model.keras`.


In [ ]:
def training_callbacks(checkpoint_path):
    return [
        tf.keras.callbacks.ModelCheckpoint(str(checkpoint_path), monitor="val_loss", save_best_only=True),
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
        tf.keras.callbacks.TerminateOnNaN(),
    ]


def save_history(history, name):
    if not np.isfinite(history.history["val_loss"]).any():
        raise RuntimeError("Training produced no finite validation loss; do not export predictions")
    with (ARTIFACT_DIR / name).open("w") as file:
        json.dump({key: [float(value) for value in values] for key, values in history.history.items()}, file, indent=2)


pretrain_history = model.fit(
    make_dataset(train_indices, training=True),
    validation_data=make_dataset(validation_indices),
    epochs=PRETRAIN_EPOCHS, callbacks=training_callbacks(PRETRAINED_PATH),
    verbose=2, shuffle=False,
)
save_history(pretrain_history, "pretrain_history.json")
print("Best pre-training validation macro log2 MSE:", min(pretrain_history.history["val_loss"]))
print("Saved pre-trained model:", PRETRAINED_PATH)


## Phase 2: Fine-tune on the Provided Data

Fine-tuning starts from the best pre-trained weights. It uses only provided-data training rows, and a new Adam optimiser with a lower learning rate so the pre-trained features are adjusted rather than overwritten. Early stopping and checkpoint selection use the provided-data validation rows, which match the test distribution. The per-setting weights are recomputed for this subset, so the 9 settings again count equally.


In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=FINETUNE_LEARNING_RATE),
    loss="mean_squared_error",
    weighted_metrics=[],
)
finetune_history = model.fit(
    make_dataset(provided_train_indices, training=True),
    validation_data=make_dataset(provided_validation_indices),
    epochs=FINETUNE_EPOCHS, callbacks=training_callbacks(MODEL_PATH),
    verbose=2, shuffle=False,
)
save_history(finetune_history, "finetune_history.json")
print("Best fine-tuning validation macro log2 MSE (provided rows):", min(finetune_history.history["val_loss"]))
print("Saved fine-tuned model:", MODEL_PATH)


## Test the DNN

Inference uses saved models with `training=False`. The only numerical post-processing is `np.exp2`, which converts log₂ predictions to m-heights.

The pre-trained model and the fine-tuned model are both evaluated on the provided-data validation and test rows. This shows what fine-tuning adds. The fine-tuned model is also evaluated on all extended test rows.


In [ ]:
def load_prediction_model(model_path=MODEL_PATH):
    with tf.device("/CPU:0"):
        return tf.keras.models.load_model(str(model_path), compile=False)


def model_log2_batch(network, batch):
    with tf.device("/CPU:0"):
        values = np.asarray(network(batch, training=False)).reshape(-1).astype(np.float64)
    if not np.isfinite(values).all() or np.any(values < 0) or np.any(values > MAX_LOG2_HEIGHT):
        raise ValueError("Model returned an invalid log2 m-height; predictions were not saved")
    return values


def predict_heights(network, samples):
    predictions = np.empty(len(samples), dtype=np.float64)
    for start in range(0, len(samples), INFERENCE_BATCH_SIZE):
        stop = min(start + INFERENCE_BATCH_SIZE, len(samples))
        log_values = model_log2_batch(network, pack_features(samples[start:stop]))
        predictions[start:stop] = np.exp2(log_values)
    if not np.isfinite(predictions).all() or np.any(predictions < 1):
        raise ValueError("Predictions must be finite and at least 1")
    return predictions


def save_prediction_list(path, predictions):
    predictions = np.asarray(predictions, dtype=np.float64)
    if predictions.ndim != 1 or not np.isfinite(predictions).all() or np.any(predictions < 1):
        raise ValueError("Expected a one-dimensional list of finite predictions >= 1")
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    with temporary.open("wb") as file:
        pickle.dump(predictions.tolist(), file, protocol=pickle.HIGHEST_PROTOCOL)
    temporary.replace(path)


In [ ]:
def report_costs(name, indices, inference_model):
    predicted_log2 = np.empty(len(indices), dtype=np.float64)
    for start in range(0, len(indices), INFERENCE_BATCH_SIZE):
        selected = indices[start:start + INFERENCE_BATCH_SIZE]
        batch = {key: values[selected] for key, values in inputs.items()}
        predicted_log2[start:start + len(selected)] = model_log2_batch(inference_model, batch)
    # Use the original float64 labels for reporting the specified cost.
    errors = (np.log2(raw_targets[indices]) - predicted_log2) ** 2
    rows = []
    print(f"\n{name}: per-setting squared log2 error")
    for group_id, setting in enumerate(PARAMETER_SETTINGS):
        mask = setting_ids[indices] == group_id
        cost = float(errors[mask].mean())
        rows.append({"n": setting[0], "k": setting[1], "m": setting[2],
                     "count": int(mask.sum()), "sigma": cost})
        print(f"{setting}: count={mask.sum():,}, sigma={cost:.6f}")
    macro = float(np.mean([row["sigma"] for row in rows]))
    print(f"Equal-setting mean cost: {macro:.6f}")
    print(f"Row-weighted mean cost: {errors.mean():.6f}")
    return {"per_setting": rows, "macro_sigma": macro, "row_mean_sigma": float(errors.mean())}

pretrained_model = load_prediction_model(PRETRAINED_PATH)
inference_model = load_prediction_model(MODEL_PATH)
cost_report = {
    "pretrained": {
        "validation_provided": report_costs("Pre-trained, validation (provided rows)", provided_validation_indices, pretrained_model),
        "local_test_provided": report_costs("Pre-trained, held-out test (provided rows)", provided_test_indices, pretrained_model),
    },
    "finetuned": {
        "validation_provided": report_costs("Fine-tuned, validation (provided rows)", provided_validation_indices, inference_model),
        "local_test_provided": report_costs("Fine-tuned, held-out test (provided rows)", provided_test_indices, inference_model),
        "local_test_extended": report_costs("Fine-tuned, held-out test (all extended rows)", test_indices, inference_model),
    },
}
with (ARTIFACT_DIR / "costs.json").open("w") as file:
    json.dump(cost_report, file, indent=2)


## Compute σ from Prediction Files

This cell computes the project's cost exactly as the specification defines it, from three lists:
- the input samples `[n, k, m, P]`;
- the true m-heights;
- the predicted m-heights.

For each parameter setting $(n,k,m)$:

$$\sigma_{n,k,m} = \frac{1}{N_{n,k,m}} \sum_t \left(\log_2 y^{[t]} - \log_2 \hat y^{[t]}\right)^2$$

Here the fine-tuned model's predictions for the provided-data held-out test rows go through `predict_heights` and `save_prediction_list`, are read back from the pickle, and are then scored.


In [ ]:
def compute_sigma(samples, true_heights, predicted_heights):
    true_heights = np.asarray(true_heights, dtype=np.float64)
    predicted_heights = np.asarray(predicted_heights, dtype=np.float64)
    if not len(samples) == len(true_heights) == len(predicted_heights):
        raise ValueError("Samples, true heights and predictions must have the same length")
    if not np.isfinite(predicted_heights).all() or np.any(predicted_heights < 1):
        raise ValueError("Every predicted m-height must be finite and at least 1")
    costs = (np.log2(true_heights) - np.log2(predicted_heights)) ** 2
    settings = [(int(n), int(k), int(m)) for n, k, m, _ in samples]
    sigma = {}
    for setting in PARAMETER_SETTINGS:
        mask = np.array([s == setting for s in settings])
        if mask.any():
            sigma[setting] = float(costs[mask].mean())
            print(f"sigma{setting} = {sigma[setting]:.6f}  (N = {mask.sum():,})")
    print(f"Equal-setting mean sigma: {np.mean(list(sigma.values())):.6f}")
    print(f"Row-weighted mean sigma:  {costs.mean():.6f}")
    return sigma


test_samples = [features[i] for i in provided_test_indices]
test_true_heights = [labels[i] for i in provided_test_indices]
prediction_path = ARTIFACT_DIR / "local_test_predictions.pkl"
save_prediction_list(prediction_path, predict_heights(inference_model, test_samples))
with prediction_path.open("rb") as file:
    test_predictions = pickle.load(file)
test_sigma = compute_sigma(test_samples, test_true_heights, test_predictions)
